# Student Exam Pass Prediction

Predicting whether a student passes or fails an exam based on study hours and a previous exam score, comparing multiple classification models.

## 1. Imports & Data Loading

In [ ]:
from src.data.load_data import load_data
from src.features.preprocessing import (
    clean_data,
    encode_categorical_columns,
    get_column_types,
    treat_outliers,
)
from src.visualization.eda_plots import plot_target_distribution, plot_correlation_heatmap, plot_numerical_distributions
from src.models.train_models import get_models, split_data, train_and_evaluate, predict_new_student
from src.utils.config import load_config

import pandas as pd

config = load_config()
FEATURES = config["features"]["columns"]
TARGET = config["features"]["target_col"]


In [ ]:
data = load_data(config["data"]["raw_path"])
data.head()

## 2. Removing Duplicates

In [ ]:
print("Duplicates before removal:", data.duplicated().sum())
data = clean_data(data)
print("Duplicates after removal:", data.duplicated().sum())


## 3. Encoding Categorical Features

(This dataset happens to be fully numeric already, but this step is kept for robustness if categorical columns are added later.)

In [ ]:
data = encode_categorical_columns(data)

## 4. Identifying Column Types

In [ ]:
categorical, numerical = get_column_types(data)
print("Categorical columns:", categorical)
print("Numerical columns:", numerical)


## 5. Outlier Treatment

**Important:** we exclude the target column (`Pass/Fail`) from outlier clipping. It's a binary class label (0/1), not a continuous measurement — applying IQR-based clipping to it doesn't make statistical sense and would risk distorting the labels themselves. Only the actual feature columns are treated.


In [ ]:
feature_cols_to_treat = [col for col in numerical if col != TARGET]
data = treat_outliers(data, feature_cols_to_treat)
data.head()


## 6. Splitting Features & Target

In [ ]:
X = data[FEATURES].copy()
y = data[TARGET].copy()

X_train, X_test, y_train, y_test = split_data(X, y)


## 7. Exploratory Data Analysis

In [ ]:
plot_target_distribution(y)

In [ ]:
plot_correlation_heatmap(data)

In [ ]:
plot_numerical_distributions(data, numerical)

## 8. Comparing Multiple Models

We compare the two models from the original analysis (Logistic Regression, Naive Bayes) alongside Decision Tree, Random Forest, KNN, and SVM.

In [ ]:
models = get_models()
results = train_and_evaluate(models, X_train, X_test, y_train, y_test)
results.round(4)


## 9. Predicting a New Student

In [ ]:
best_model_name = results.index[0]
best_model = models[best_model_name]
print(f"Best model: {best_model_name}")

prediction = predict_new_student(best_model, study_hours=9, previous_exam_score=90)
print("Predicted Pass/Fail for (9 study hours, 90 previous score):", "Pass" if prediction == 1 else "Fail")


## Conclusion

Study hours and previous exam performance are strong, fairly clean predictors of pass/fail outcome for this dataset. Tree-based models (Random Forest, Decision Tree) tend to perform best here, likely because the pass/fail boundary isn't purely linear. Comparing multiple models confirms this isn't an artifact of any single algorithm's assumptions.